<a href="https://colab.research.google.com/github/annaathafunnisa770-pixel/Tugas-Data-Wrangling-1/blob/main/Tugas1_Anna_Althafun_Nisa_25200003.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TUGAS MANDIRI DATA WRANGLING PERTEMUAN 2
**Nama:** Anna Athafun Nisa  
**NIM:** [25200003]  

---

## BAGIAN A: Analisis Konseptual & Tata Kelola Data

### 1. Tabel Pemetaan Kebutuhan Bisnis ke Teknis Data
| Kebutuhan Bisnis | Atribut / Kolom | Tipe Data Teknis | Sumber Asal Data | Catatan & Format |
| :--- | :--- | :--- | :--- | :--- |
| **Identifikasi Transaksi** | `order_id` | `String` / `Object` | Divisi POS (File CSV) | Primary key transaksi, wajib unik |
| **Identifikasi Pelanggan** | `customer_id` | `String` / `Object` | Divisi POS (CSV) & CRM (SQL) | Foreign key untuk penggabungan tabel |
| **Tanggal Transaksi** | `order_date` | `Datetime` | Divisi POS (File CSV) | Diparse ke datetime (YYYY-MM-DD) |
| **Nilai Transaksi** | `total_amount` | `Float64` / `Int64` | Divisi POS (File CSV) | Untuk agregasi & validasi (> 0) |
| **Penggunaan Diskon** | `coupon_code` | `String` | Divisi POS (File CSV) | Kode kupon diskon akhir bulan |
| **Profil & Keanggotaan** | `customer_name`, `membership_tier` | `String` | Divisi CRM (Database SQL) | Mengukur level loyalitas pelanggan |
| **Demografi Pelanggan** | `city` | `String` | Divisi CRM (Database SQL) | Kota asal/domisili pelanggan |
| **Status Pengiriman** | `tracking_number`, `delivery_status` | `String` | Mitra Ekspedisi (REST API) | Informasi resi dan status paket |

---

### 2. Tabel Matriks Peran RACI (Data Governance)
* **R (Responsible):** Pihak yang mengeksekusi/mengerjakan tugas.
* **A (Accountable):** Pihak yang bertanggung jawab penuh dan memiliki wewenang keputusan.
* **C (Consulted):** Pihak yang memberikan masukan teknis atau konsultasi.
* **I (Informed):** Pihak yang menerima laporan hasil/pembaruan.

| Aktivitas / Tahapan Kerja | Head of Business Strategy (Data Owner) | Data Management Office / Data Steward | Data Wrangler / Engineer (Kamu) | Business Intelligence Analyst (Data User) |
| :--- | :---: | :---: | :---: | :---: |
| **Penerjemahan Kebutuhan Bisnis ke Data** | **Accountable (A)** | Consulted (C) | Responsible (R) | Informed (I) |
| **Penetapan Aturan Kualitas & Tata Kelola Data** | Accountable (A) | **Responsible (R)** | Consulted (C) | Informed (I) |
| **Ekstraksi, Ingestion & Integrasi Data Multi-Sumber** | Informed (I) | Consulted (C) | **Responsible (R) / Accountable (A)** | Informed (I) |
| **Otomasi Kamus Data & Pengujian Validasi (Assert)** | Informed (I) | **Accountable (A)** | **Responsible (R)** | Informed (I) |
| **Analisis Performa Kampanye Diskon & Logistik** | Accountable (A) | Informed (I) | Informed (I) | **Responsible (R)** |

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta

print("[INFO] Membuat data dummy penunjang (POS & CRM)...")

# 1. Generate Data Transaksi POS (CSV)
np.random.seed(42)
n_records = 50

order_ids = [f"ORD-{1001 + i}" for i in range(n_records)]
customer_ids = [f"CUST-{np.random.randint(101, 115)}" for _ in range(n_records)]

end_date = datetime.now()
start_date = end_date - timedelta(days=30)
random_dates = [start_date + timedelta(days=int(np.random.randint(0, 30))) for _ in range(n_records)]

total_amounts = np.random.choice([50000, 120000, 250000, 450000, 750000, 1200000], size=n_records)
coupons = np.random.choice(["DISKON30", "GAJIANMEGAMURAH", "CASHBACK10", None], size=n_records, p=[0.3, 0.2, 0.2, 0.3])

df_pos_dummy = pd.DataFrame({
    "order_id": order_ids,
    "customer_id": customer_ids,
    "order_date": random_dates,
    "total_amount": total_amounts,
    "coupon_code": coupons
})
df_pos_dummy.to_csv("transaksi_pos.csv", index=False)

# 2. Generate Data CRM (Database SQLite)
customers_data = {
    "customer_id": [f"CUST-{i}" for i in range(101, 116)],
    "customer_name": [
        "Andi Pratama", "Budi Santoso", "Citra Dewi", "Dian Sastro", "Eko Prasetyo",
        "Fani Rahmawati", "Gita Gutawa", "Hendra Wijaya", "Indah Permata", "Joko Widodo",
        "Kurnia Meiga", "Lestari Putri", "Mawan Setiawan", "Nia Ramadhani", "Oki Setiana"
    ],
    "membership_tier": np.random.choice(["Bronze", "Silver", "Gold", "Platinum"], size=15),
    "city": np.random.choice(["Jakarta", "Surabaya", "Bandung", "Semarang", "Yogyakarta"], size=15),
    "is_active": [1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1]
}

df_crm_dummy = pd.DataFrame(customers_data)
conn = sqlite3.connect("crm_loyalty.db")
df_crm_dummy.to_sql("customers", conn, if_exists="replace", index=False)
conn.close()

print("[SUCCESS] Berhasil membuat file 'transaksi_pos.csv' dan 'crm_loyalty.db'!")

[INFO] Membuat data dummy penunjang (POS & CRM)...
[SUCCESS] Berhasil membuat file 'transaksi_pos.csv' dan 'crm_loyalty.db'!


In [ ]:
import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine

# 1. EKSTRAKSI FILE CSV
print("[INFO] 1. Mengimpor berkas transaksi POS...")
df_transaksi = pd.read_csv(
    "transaksi_pos.csv",
    dtype={"order_id": str, "customer_id": str},
    parse_dates=["order_date"]
)

# 2. EKSTRAKSI DATABASE SQL (CRM)
print("[INFO] 2. Menghubungkan ke database CRM...")
engine = create_engine("sqlite:///crm_loyalty.db")
query_crm = """
SELECT customer_id, customer_name, membership_tier, city
FROM customers
WHERE is_active = 1
"""
df_crm = pd.read_sql(query_crm, con=engine)

# 3. KONSUMSI REST API (LOGISTIK)
print("[INFO] 3. Menarik data pengiriman dari REST API...")
api_url = "https://api.mockfly.dev/mocks/dummy-logistik/deliveries"

try:
    response = requests.get(api_url, timeout=10)
    if response.status_code == 200:
        data_logistik = response.json()
        df_logistik = pd.json_normalize(data_logistik["results"])
    else:
        df_logistik = pd.DataFrame(columns=["order_id", "tracking_number", "delivery_status"])
except Exception as e:
    df_logistik = pd.DataFrame(columns=["order_id", "tracking_number", "delivery_status"])

# 4. INTEGRASI DATA MULTI-SUMBER (MERGING)
print("[INFO] 4. Melakukan integrasi data multi-sumber...")
df_master = df_transaksi.merge(df_crm, on="customer_id", how="inner")
df_master_analisis = df_master.merge(df_logistik, on="order_id", how="left")

print(f"[SUCCESS] Integrasi berhasil! Total baris data master: {len(df_master_analisis)}")
df_master_analisis.head()

[INFO] 1. Mengimpor berkas transaksi POS...
[INFO] 2. Menghubungkan ke database CRM...
[INFO] 3. Menarik data pengiriman dari REST API...
[INFO] 4. Melakukan integrasi data multi-sumber...
[SUCCESS] Integrasi berhasil! Total baris data master: 45


,order_id,customer_id,order_date,total_amount,coupon_code,customer_name,membership_tier,city,tracking_number,delivery_status
0,ORD-1001,CUST-107,2026-09-03 03:20:26.077152,1200000,NaN,Gita Gutawa,Bronze,Surabaya,NaN,NaN
1,ORD-1002,CUST-104,2026-09-05 03:20:26.077152,450000,CASHBACK10,Dian Sastro,Bronze,Surabaya,NaN,NaN
2,ORD-1004,CUST-111,2026-09-07 03:20:26.077152,1200000,GAJIANMEGAMURAH,Kurnia Meiga,Silver,Jakarta,NaN,NaN
3,ORD-1005,CUST-108,2026-09-21 03:20:26.077152,750000,CASHBACK10,Hendra Wijaya,Gold,Semarang,NaN,NaN
4,ORD-1007,CUST-105,2026-09-07 03:20:26.077152,120000,DISKON30,Eko Prasetyo,Bronze,Yogyakarta,NaN,NaN


### Alasan Pemilihan Tipe Join:
1. **`df_transaksi` merge `df_crm` (INNER JOIN):**
   Menggunakan `inner` join untuk memastikan kita hanya menganalisis transaksi dari pelanggan yang aktif dan sah terdaftar pada sistem CRM. Transaksi dari akun non-aktif disaring dari analisis.
2. **`df_master` merge `df_logistik` (LEFT JOIN):**
   Menggunakan `left` join agar seluruh data transaksi utama dari POS/CRM tidak hilang, meskipun status pengirimannya belum terupdate atau belum masuk di sistem API logistik.

In [ ]:
# 1. OTOMASI KAMUS DATA (DATA DICTIONARY)
def generate_data_dictionary(df):
    dict_list = []
    for col in df.columns:
        dict_list.append({
            "Nama Kolom": col,
            "Tipe Data": str(df[col].dtype),
            "Missing Values": df[col].isnull().sum(),
            "Missing (%)": round((df[col].isnull().sum() / len(df)) * 100, 2),
            "Jumlah Unik": df[col].nunique(),
            "Contoh Nilai": str(df[col].dropna().iloc[0]) if not df[col].dropna().empty else "-"
        })
    return pd.DataFrame(dict_list)

kamus_df = generate_data_dictionary(df_master_analisis)
kamus_df.to_csv("kamus_data_hasil.csv", index=False)
print("[INFO] Kamus data berhasil disimpan ke 'kamus_data_hasil.csv'.")

# 2. VALIDASI KUALITAS DATA (ASSERTIONS)
print("[INFO] Memulai proses validasi data...")

# Assertion 1: Tidak ada order_id duplikat
assert df_master_analisis["order_id"].is_unique, "Fatal: Terdeteksi order_id duplikat!"

# Assertion 2: Transaksi tidak ada yang negatif
assert (df_master_analisis["total_amount"] >= 0).all(), "Fatal: Ada nilai transaksi negatif!"

# Assertion 3: Tanggal pesanan tidak melebihi hari ini
assert (df_master_analisis["order_date"] <= pd.Timestamp.now()).all(), "Fatal: Terdeteksi tanggal pesanan di masa depan!"

print("[SUCCESS] Seluruh validasi data lulus tanpa error!")

[INFO] Kamus data berhasil disimpan ke 'kamus_data_hasil.csv'.
[INFO] Memulai proses validasi data...
[SUCCESS] Seluruh validasi data lulus tanpa error!
